# Formulaic prose → natural prose: seq2seq PoC

This notebook fine-tunes `google/flan-t5-small` on the fixed ARB train, validation, and test splits. Upload the `arb_dataset.zip` produced by `src.prepare_arb_dataset`.

In [ ]:
!pip -q install 'transformers>=4.48,<5' 'datasets>=3.2,<4' 'accelerate>=1.2,<2' 'evaluate>=0.4,<1' 'sentence-transformers>=3.3,<4' 'sacrebleu>=2.4,<3' 'sacremoses>=0.1,<1'

In [ ]:
import json, os, re, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer,
    DataCollatorForSeq2Seq, EarlyStoppingCallback, Seq2SeqTrainer,
    Seq2SeqTrainingArguments)

SEED = 42
MODEL_NAME = 'google/flan-t5-small'
OUTPUT_DIR = Path('artifacts/flan-t5-small-naturalizer')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('device:', 'cuda' if torch.cuda.is_available() else 'cpu')
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab before training.'

## Load paired data
Upload `arb_dataset.zip`. The notebook uses its existing group-safe splits and checks that no source group appears in more than one split.

In [ ]:
from google.colab import files
uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
assert len(archives) == 1, 'Upload exactly one dataset ZIP.'
shutil.unpack_archive(archives[0], '.')
data_dir = Path('data/processed/arb')
expected = {name: data_dir / f'{name}.csv' for name in ('train', 'validation', 'test')}
missing = [str(path) for path in expected.values() if not path.exists()]
assert not missing, f'Missing prepared splits: {missing}'
splits = {name: pd.read_csv(path) for name, path in expected.items()}
required = {'source', 'target', 'group_id', 'pair_type'}
for name, frame in splits.items():
    absent = required - set(frame.columns)
    assert not absent, f'{name} is missing columns: {absent}'
    assert not frame[list(required)].isna().any().any(), f'{name} contains missing required values.'
    assert not frame.duplicated(['source', 'target']).any(), f'{name} contains duplicate pairs.'

In [ ]:
group_sets = {name: set(frame.group_id) for name, frame in splits.items()}
assert not group_sets['train'] & group_sets['validation']
assert not group_sets['train'] & group_sets['test']
assert not group_sets['validation'] & group_sets['test']
train_df, val_df, test_df = splits['train'], splits['validation'], splits['test']
summary = {name: {'rows': len(frame), 'groups': frame.group_id.nunique(),
                  'types': frame.pair_type.value_counts().to_dict()}
           for name, frame in splits.items()}
summary

## Tokenize and fine-tune

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
prefix = 'Rewrite this in natural, direct prose without changing its meaning: '
truncation_stats = {}
for name, frame in splits.items():
    source_ids = tokenizer([prefix + x for x in frame.source], truncation=False).input_ids
    target_ids = tokenizer(frame.target.tolist(), truncation=False).input_ids
    truncation_stats[name] = {
        'source_over_512': sum(len(ids) > 512 for ids in source_ids),
        'target_over_512': sum(len(ids) > 512 for ids in target_ids),
    }
print('truncation audit:', truncation_stats)
raw = DatasetDict({k: Dataset.from_pandas(v[['source', 'target']], preserve_index=False) for k, v in splits.items()})

def tokenize(batch):
    inputs = tokenizer([prefix + x for x in batch['source']], max_length=512, truncation=True)
    labels = tokenizer(text_target=batch['target'], max_length=512, truncation=True)
    inputs['labels'] = labels['input_ids']
    return inputs

tokenized = raw.map(tokenize, batched=True, remove_columns=raw['train'].column_names)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR), learning_rate=5e-5, warmup_ratio=0.05,
    per_device_train_batch_size=4, per_device_eval_batch_size=8,
    gradient_accumulation_steps=1, num_train_epochs=5, weight_decay=0.01,
    eval_strategy='epoch', save_strategy='epoch', save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model='eval_loss', greater_is_better=False,
    predict_with_generate=True, generation_max_length=512,
    logging_strategy='epoch', save_only_model=True,
    fp16=False, bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to='none', seed=SEED,
)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=tokenized['train'],
    eval_dataset=tokenized['validation'], processing_class=tokenizer, data_collator=collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()
trainer.save_model(str(OUTPUT_DIR)); tokenizer.save_pretrained(str(OUTPUT_DIR))

## Held-out evaluation
SARI measures editing against the reference. Embedding similarity and entity/number preservation are safety indicators, not proof of equivalent meaning. The final decision requires blind human review.

In [ ]:
pred = trainer.predict(tokenized['test'])
pred_ids = pred.predictions[0] if isinstance(pred.predictions, tuple) else pred.predictions
outputs = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
result_columns = ['group_id', 'source', 'target', 'pair_type', 'source_dataset', 'generator_model']
results = test_df[result_columns].reset_index(drop=True).copy()
results['prediction'] = outputs
rewrite_results = results[results.pair_type == 'rewrite'].reset_index(drop=True)
identity_results = results[results.pair_type == 'identity'].reset_index(drop=True)
results

In [ ]:
import evaluate
from sentence_transformers import SentenceTransformer

sari = evaluate.load('sari')
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def semantic_similarity(sources, candidates):
    a = embedder.encode(list(sources), normalize_embeddings=True)
    b = embedder.encode(list(candidates), normalize_embeddings=True)
    return np.sum(a * b, axis=1)

def protected_tokens(text):
    # Review this heuristic manually: capitalization is an imperfect proxy for names.
    numbers = re.findall(r'(?<!\w)[+-]?(?:\d[\d,.]*%?)(?!\w)', text)
    names = []
    for match in re.finditer(r'\b[A-Z][A-Za-z0-9_-]+\b', text):
        before = text[:match.start()].rstrip()
        if before and before[-1] not in '.!?':
            names.append(match.group())
    return set(numbers + names)

def preservation_rate(sources, candidates):
    scores = []
    for src, cand in zip(sources, candidates):
        tokens = protected_tokens(src)
        scores.append(1.0 if not tokens else len(tokens & protected_tokens(cand)) / len(tokens))
    return np.array(scores)

def normalized(text):
    return re.sub(r'\s+', ' ', text).strip()

def score(subset, frame, label, candidates):
    candidates = list(candidates)
    return {
        'subset': subset,
        'system': label,
        'examples': len(frame),
        'SARI': sari.compute(sources=frame.source.tolist(), predictions=candidates, references=[[x] for x in frame.target])['sari'],
        'source_similarity': semantic_similarity(frame.source, candidates).mean(),
        'protected_token_recall': preservation_rate(frame.source, candidates).mean(),
        'exact_copy_rate': np.mean([normalized(a) == normalized(b) for a, b in zip(frame.source, candidates)]),
    }

metrics = pd.DataFrame([
    score('rewrite', rewrite_results, 'identity baseline', rewrite_results.source),
    score('rewrite', rewrite_results, 'fine-tuned model', rewrite_results.prediction),
    score('identity', identity_results, 'identity baseline', identity_results.source),
    score('identity', identity_results, 'fine-tuned model', identity_results.prediction),
])
metrics

In [ ]:
# Export predictions plus a blinded A/B sheet for human review.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results['source_similarity'] = semantic_similarity(results.source, results.prediction)
results['protected_token_recall'] = preservation_rate(results.source, results.prediction)
results.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False)
rewrite_results.to_csv(OUTPUT_DIR / 'rewrite_test_predictions.csv', index=False)
identity_results.to_csv(OUTPUT_DIR / 'identity_test_predictions.csv', index=False)
metrics.to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
(OUTPUT_DIR / 'truncation_report.json').write_text(json.dumps(truncation_stats, indent=2) + '\n')
pd.DataFrame(trainer.state.log_history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)
rng = np.random.default_rng(SEED)
model_is_a = rng.random(len(rewrite_results)) < 0.5
review = pd.DataFrame({
    'item_id': np.arange(len(rewrite_results)), 'source': rewrite_results.source,
    'option_a': np.where(model_is_a, rewrite_results.prediction, rewrite_results.source),
    'option_b': np.where(model_is_a, rewrite_results.source, rewrite_results.prediction),
    'naturalness_winner_A_B_TIE': '', 'a_preserves_meaning_Y_N': '',
    'b_preserves_meaning_Y_N': '', 'notes': ''})
key = pd.DataFrame({'item_id': review.item_id, 'model_option': np.where(model_is_a, 'A', 'B')})
review.to_csv(OUTPUT_DIR / 'blind_review.csv', index=False)
key.to_csv(OUTPUT_DIR / 'blind_review_key.csv', index=False)
identity_review = identity_results[['group_id', 'source', 'prediction']].copy()
identity_review['changed'] = [normalized(a) != normalized(b) for a, b in zip(identity_review.source, identity_review.prediction)]
identity_review['change_was_necessary_Y_N'] = ''
identity_review['meaning_preserved_Y_N'] = ''
identity_review['notes'] = ''
identity_review.to_csv(OUTPUT_DIR / 'identity_review.csv', index=False)
# The root model is already saved. Checkpoints only duplicate it in the download.
for checkpoint in OUTPUT_DIR.glob('checkpoint-*'):
    shutil.rmtree(checkpoint)
shutil.make_archive('naturalizer_artifacts', 'zip', 'artifacts')
files.download('naturalizer_artifacts.zip')